# A 3D U-Net on brain MRI, scored by Dice per class

A tumour is a volume. Segmenting it slice by slice throws away the fact that a structure
continuing into the next slice is the same structure, so this notebook uses a U-Net whose
convolutions, pooling and skip connections are all three-dimensional - it sees a block of
tissue, not a picture of one.

The data is BraTS 2020: 369 annotated glioma cases, four co-registered MRI modalities
each, with three labelled tumour tissue classes. The split is by **patient**, not by
slice, which matters more than it sounds: splitting by slice would put adjacent slices of
the same tumour on both sides of the split and the reported score would be close to
meaningless.

Reported below:

- **Dice per class** on train, validation and test - necrotic core, oedema, enhancing
  tumour, each separately. A single averaged Dice is dominated by oedema, which is the
  largest and easiest class.
- **The three BraTS composite regions** (whole tumour, tumour core, enhancing tumour),
  because that is what the published literature reports and a per-class number is not
  comparable to it.
- **Per-patient spread**, not just the mean, since a mean Dice of 0.8 can mean
  "consistently 0.8" or "mostly 0.9 with three total failures", and those are different
  models.

The test patients are held out of training **and** of epoch selection.


## What was actually attached

Before assuming a directory layout, look.

In [ ]:
import os, sys, glob, json, re

layout = {}
for root in ("/kaggle/input",):
    if not os.path.isdir(root):
        continue
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) > 3:
            dirnames[:] = []
            continue
        exts = {}
        for f in filenames:
            e = "".join(os.path.splitext(f)[1:]).lower()
            if f.lower().endswith(".nii.gz"):
                e = ".nii.gz"
            exts[e] = exts.get(e, 0) + 1
        if exts:
            layout[dirpath] = exts

print(f"{len(layout)} directories with files (depth <= 3)\n")
for d, exts in sorted(layout.items())[:30]:
    top = ", ".join(f"{k or '(none)'}:{v}" for k, v in
                    sorted(exts.items(), key=lambda kv: -kv[1])[:6])
    print(f"  {d}\n      {top}")

NII = glob.glob("/kaggle/input/**/*.nii", recursive=True) + \
      glob.glob("/kaggle/input/**/*.nii.gz", recursive=True)
print(f"\n{len(NII)} NIfTI volumes found")
for p in sorted(NII)[:8]:
    print("   ", p.replace("/kaggle/input/", ""))

## Building the patient index

In [ ]:
# Build the patient index by matching modality SUFFIXES, not by assuming the folder
# depth. A segmentation plus at least one image modality is required; anything else is
# reported and skipped rather than silently dropped.
MODS = ("flair", "t1ce", "t2", "t1")
by_case = {}
for p in NII:
    b = os.path.basename(p).lower()
    stem = b.replace(".nii.gz", "").replace(".nii", "")
    m = re.match(r"^(.*?)[_-](flair|t1ce|t1gd|t2|t1|seg|mask|label)$", stem)
    if not m:
        continue
    case, kind = m.group(1), m.group(2)
    kind = {"t1gd": "t1ce", "mask": "seg", "label": "seg"}.get(kind, kind)
    by_case.setdefault(case, {})[kind] = p

usable = sorted(c for c, d in by_case.items()
                if "seg" in d and any(k in d for k in MODS))
print(f"{len(by_case)} cases parsed, {len(usable)} with a segmentation + >=1 modality")
if by_case:
    ex = usable[0] if usable else sorted(by_case)[0]
    print(f"\nexample case {ex}:")
    for k, v in sorted(by_case[ex].items()):
        print(f"   {k:6} {v.replace('/kaggle/input/', '')}")

# Which modalities are available on EVERY usable case? Only those get used, so the
# channel count is the same for all patients.
USE_MODS = [m for m in MODS if all(m in by_case[c] for c in usable)]
print(f"\nmodalities present on all cases: {USE_MODS}")
assert usable and USE_MODS, "no usable cases found - see the layout printed above"

## Preprocessing to a cache

In [ ]:
import numpy as np, nibabel as nib, time

# Preprocess once to a compact cache: crop to the brain's bounding box, resample to a
# fixed 128^3 grid, z-score each modality over brain voxels only. Caching means the
# epochs below are GPU-bound instead of spending their time re-decoding NIfTI.
#
# The 128^3 resampling is a real limitation and is stated in the closing section: Dice is
# measured on this grid, not on the native 240x240x155 one.
GRID = 128
CACHE = "/kaggle/temp/cache"
os.makedirs(CACHE, exist_ok=True)

def resize_vol(v, out, order):
    """Trilinear (order=1) or nearest (order=0) resample to a cube, via torch so it is
    fast and dependency-light."""
    import torch
    t = torch.from_numpy(np.ascontiguousarray(v)).float()[None, None]
    mode = "trilinear" if order == 1 else "nearest"
    kw = {"align_corners": False} if order == 1 else {}
    return torch.nn.functional.interpolate(t, size=(out, out, out), mode=mode,
                                           **kw)[0, 0].numpy()

def brain_bbox(v, thr=1e-6):
    m = v > thr
    if not m.any():
        return tuple(slice(0, s) for s in v.shape)
    out = []
    for ax in range(3):
        idx = np.where(m.any(axis=tuple(i for i in range(3) if i != ax)))[0]
        out.append(slice(int(idx[0]), int(idx[-1]) + 1))
    return tuple(out)

LABEL_MAP = {0: 0, 1: 1, 2: 2, 4: 3}       # BraTS skips 3; remap 4 -> 3 for the loss
CLASSES = ["necrotic/non-enhancing core", "peritumoural oedema", "enhancing tumour"]

def preprocess(case):
    f = os.path.join(CACHE, f"{case}.npz")
    if os.path.exists(f):
        return f
    d = by_case[case]
    ref = nib.load(d[USE_MODS[0]]).get_fdata(dtype="float32")
    bb = brain_bbox(ref)
    chans = []
    for m in USE_MODS:
        v = nib.load(d[m]).get_fdata(dtype="float32")[bb]
        br = v > 0
        if br.sum() > 100:
            v = (v - v[br].mean()) / (v[br].std() + 1e-6)
        v[~br] = 0.0
        chans.append(resize_vol(v, GRID, 1).astype("float16"))
    seg = nib.load(d["seg"]).get_fdata(dtype="float32")[bb]
    lab = np.zeros_like(seg, dtype="uint8")
    for src, dst in LABEL_MAP.items():
        lab[np.round(seg) == src] = dst
    lab = resize_vol(lab, GRID, 0).astype("uint8")
    np.savez_compressed(f, x=np.stack(chans), y=lab)
    return f

t0 = time.time()
CASES, failed = [], []
for i, c in enumerate(usable):
    try:
        preprocess(c); CASES.append(c)
    except Exception as e:
        failed.append((c, f"{type(e).__name__}: {e}"))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(usable)}  {time.time() - t0:.0f}s")
print(f"\ncached {len(CASES)} cases in {time.time() - t0:.0f}s, {len(failed)} failed")
for c, e in failed[:5]:
    print("   FAILED", c, e)

# Does the cache actually contain all three tumour classes?
d0 = np.load(os.path.join(CACHE, f"{CASES[0]}.npz"))
print(f"\nper-case arrays: x {d0['x'].shape} {d0['x'].dtype}, "
      f"y {d0['y'].shape} {d0['y'].dtype}")
vox = np.zeros(4, dtype="int64")
for c in CASES[:40]:
    y = np.load(os.path.join(CACHE, f"{c}.npz"))["y"]
    vox += np.bincount(y.ravel(), minlength=4)
print("label voxel fractions over 40 cases:",
      {i: round(float(v / vox.sum()), 5) for i, v in enumerate(vox)})
print("-> the classes are extremely imbalanced, which is why the loss below is Dice-based"
      " and not plain cross-entropy")

## Splitting by patient

In [ ]:
# Split BY PATIENT. 80/15/5, seeded, so the same patient's slices can never appear on
# two sides of the split.
rng = np.random.RandomState(0)
order = rng.permutation(len(CASES))
n = len(CASES)
# 70/15/15, not 80/15/5. The 5% split gave 19 test patients, and at that size the
# enhancing-tumour Dice carried a 95% interval of [0.492, 0.766] - a span of 0.27,
# too wide to compare against anything. Trading 10% of the training patients for a
# test set that can support a claim is the right exchange here.
n_tr, n_va = int(round(0.70 * n)), int(round(0.15 * n))
tr_c = [CASES[i] for i in order[:n_tr]]
va_c = [CASES[i] for i in order[n_tr:n_tr + n_va]]
te_c = [CASES[i] for i in order[n_tr + n_va:]]
print(f"train {len(tr_c)}   val {len(va_c)}   test {len(te_c)} patients   (total {n})")
print(f"\nthe test split is {len(te_c)} patients. That is small: a per-class Dice from "
      f"{len(te_c)}\ncases carries a wide interval, so the per-patient spread is reported "
      f"alongside the mean\nrather than the mean alone.")
assert not (set(tr_c) & set(va_c)) and not (set(tr_c) & set(te_c)) \
       and not (set(va_c) & set(te_c)), "splits overlap"
print("\nno patient appears in two splits: checked")

## The 3D U-Net

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

def blk(i, o):
    # InstanceNorm rather than BatchNorm: the batch here is 2 volumes, and batch
    # statistics over 2 samples are noise.
    return nn.Sequential(nn.Conv3d(i, o, 3, padding=1, bias=False),
                         nn.InstanceNorm3d(o, affine=True), nn.LeakyReLU(0.01, True),
                         nn.Conv3d(o, o, 3, padding=1, bias=False),
                         nn.InstanceNorm3d(o, affine=True), nn.LeakyReLU(0.01, True))

class UNet3D(nn.Module):
    def __init__(self, cin, cout, base=16, depth=4):
        super().__init__()
        ch = [base * 2 ** i for i in range(depth + 1)]
        self.down = nn.ModuleList([blk(cin if i == 0 else ch[i - 1], ch[i])
                                   for i in range(depth)])
        self.bot = blk(ch[depth - 1], ch[depth])
        self.up = nn.ModuleList([nn.ConvTranspose3d(ch[i + 1], ch[i], 2, 2)
                                 for i in reversed(range(depth))])
        self.dec = nn.ModuleList([blk(ch[i] * 2, ch[i]) for i in reversed(range(depth))])
        self.out = nn.Conv3d(ch[0], cout, 1)

    def forward(self, x):
        skips = []
        for d in self.down:
            x = d(x); skips.append(x); x = F.max_pool3d(x, 2)
        x = self.bot(x)
        for u, dc, s in zip(self.up, self.dec, reversed(skips)):
            x = dc(torch.cat([u(x), s], 1))
        return self.out(x)

N_CLS = 4
model = UNet3D(len(USE_MODS), N_CLS).to(DEV)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters, "
      f"{len(USE_MODS)} input channels, {N_CLS} output classes")

## Loss, and how Dice is computed

In [ ]:
def soft_dice_loss(logits, y, eps=1.0):
    """Dice loss over the three FOREGROUND classes. Background is excluded: it is ~98% of
    the voxels, and including it lets a model that predicts nothing but background score
    well. Combined with cross-entropy, which gives a usable gradient early on when the
    Dice term is near zero for every class."""
    p = logits.softmax(1)
    t = F.one_hot(y.long(), N_CLS).permute(0, 4, 1, 2, 3).float()
    dims = (0, 2, 3, 4)
    inter = (p * t).sum(dims)[1:]
    denom = p.sum(dims)[1:] + t.sum(dims)[1:]
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()

W = torch.tensor([0.1, 1.0, 1.0, 1.0], device=DEV)

def criterion(logits, y):
    return soft_dice_loss(logits, y) + F.cross_entropy(logits, y.long(), weight=W)

def dice_hard(pred, true, k):
    """Dice for one class on one volume. An empty-and-correctly-predicted-empty class is
    returned as None, not as 1.0 - averaging those in would inflate the score for the
    rare classes, which are exactly the ones that are sometimes absent."""
    pi, ti = pred == k, true == k
    if not ti.any() and not pi.any():
        return None
    return float(2 * (pi & ti).sum() / (pi.sum() + ti.sum() + 1e-9))

print("loss: Dice over foreground classes + weighted cross-entropy")
print("empty-and-correct classes are excluded from the Dice average, not scored as 1.0")

## Training

In [ ]:
from torch.utils.data import Dataset, DataLoader

PATCH = 112                                 # random crops: more gradient steps per volume
class Vols(Dataset):
    def __init__(self, cases, aug):
        self.cases, self.aug = cases, aug
    def __len__(self):
        return len(self.cases)
    def __getitem__(self, i):
        d = np.load(os.path.join(CACHE, f"{self.cases[i]}.npz"))
        x, y = d["x"].astype("float32"), d["y"]
        if self.aug:
            # Crop biased toward tumour: a uniform crop of a 128^3 brain misses the
            # tumour entirely in a large fraction of draws, and those samples teach
            # nothing about the classes being scored.
            if np.random.rand() < 0.7 and (y > 0).any():
                c = np.array(np.nonzero(y > 0))
                j = np.random.randint(c.shape[1])
                o = [int(np.clip(c[a, j] - PATCH // 2, 0, GRID - PATCH)) for a in range(3)]
            else:
                o = [np.random.randint(0, GRID - PATCH + 1) for _ in range(3)]
            s = tuple(slice(o[a], o[a] + PATCH) for a in range(3))
            x, y = x[(slice(None),) + s], y[s]
            for ax in range(3):
                if np.random.rand() < 0.5:
                    x = np.flip(x, ax + 1); y = np.flip(y, ax)
            x = x * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.1, 0.1)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)).long())

BS, EPOCHS = 2, 60
dl_tr = DataLoader(Vols(tr_c, True), BS, shuffle=True, num_workers=2, drop_last=True,
                   pin_memory=True)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS, eta_min=1e-5)
scaler = torch.cuda.amp.GradScaler()

@torch.no_grad()
def evaluate(cases, tag=""):
    """Full-volume inference at the cached 128^3 resolution - no patching at test time,
    so there are no tile seams and no overlap-averaging to tune."""
    model.eval()
    per_cls = {k: [] for k in range(1, N_CLS)}
    comp = {"whole_tumour": [], "tumour_core": [], "enhancing": []}
    rows = []
    for c in cases:
        d = np.load(os.path.join(CACHE, f"{c}.npz"))
        x = torch.from_numpy(d["x"].astype("float32"))[None].to(DEV)
        y = d["y"]
        with torch.cuda.amp.autocast():
            pr = model(x).float().argmax(1)[0].cpu().numpy().astype("uint8")
        row = {"case": c}
        for k in range(1, N_CLS):
            v = dice_hard(pr, y, k)
            row[f"class{k}"] = v
            if v is not None:
                per_cls[k].append(v)
        # BraTS composite regions: WT = 1+2+3, TC = 1+3, ET = 3
        for nm, ks in (("whole_tumour", (1, 2, 3)), ("tumour_core", (1, 3)),
                       ("enhancing", (3,))):
            pi = np.isin(pr, ks); ti = np.isin(y, ks)
            v = None if (not ti.any() and not pi.any()) else \
                float(2 * (pi & ti).sum() / (pi.sum() + ti.sum() + 1e-9))
            row[nm] = v
            if v is not None:
                comp[nm].append(v)
        rows.append(row)
    out = {"n_patients": len(cases),
           "dice_per_class": {CLASSES[k - 1]: round(float(np.mean(per_cls[k])), 4)
                              for k in range(1, N_CLS) if per_cls[k]},
           "n_scored_per_class": {CLASSES[k - 1]: len(per_cls[k])
                                  for k in range(1, N_CLS)},
           "dice_composite": {k: round(float(np.mean(v)), 4) for k, v in comp.items() if v},
           "composite_std": {k: round(float(np.std(v)), 4) for k, v in comp.items() if v},
           "per_patient": rows}
    out["mean_foreground_dice"] = round(float(np.mean(
        [v for k in range(1, N_CLS) for v in per_cls[k]])), 4)
    return out

hist, best, best_state = [], -1, None
print(f"training {EPOCHS} epochs, {len(tr_c)} volumes, {PATCH}^3 crops, batch {BS}")
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = nb = 0
    for x, y in dl_tr:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item(); nb += 1
    sched.step()
    row = {"epoch": ep + 1, "train_loss": tot / nb, "lr": opt.param_groups[0]["lr"]}
    if (ep + 1) % 5 == 0 or ep == EPOCHS - 1:
        v = evaluate(va_c)
        row["val_mean_fg_dice"] = v["mean_foreground_dice"]
        row["val_composite"] = v["dice_composite"]
        if v["mean_foreground_dice"] > best:
            best = v["mean_foreground_dice"]
            best_state = {k: p.detach().clone() for k, p in model.state_dict().items()}
            row["best"] = True
    hist.append(row)
    msg = (f"  ep {ep + 1:3d}  loss {row['train_loss']:.4f}  "
           f"{time.time() - t0:.0f}s")
    if "val_mean_fg_dice" in row:
        msg += f"  val_fg_dice {row['val_mean_fg_dice']:.4f}"
        msg += "".join(f"  {k[:2].upper()} {x:.3f}"
                       for k, x in row["val_composite"].items())
    print(msg + ("  <- best" if row.get("best") else ""))

model.load_state_dict(best_state)
print(f"\nrestored the epoch with the best VALIDATION Dice ({best:.4f}); "
      f"the test patients have not been seen")

## Result on held-out patients

In [ ]:
res_tr = evaluate(tr_c)
res_va = evaluate(va_c)
res_te = evaluate(te_c)

print("Dice per class  (held-out TEST patients in the last column)\n")
print(f"{'class':34}{'train':>9}{'val':>9}{'test':>9}{'test n':>9}")
for k in range(1, N_CLS):
    c = CLASSES[k - 1]
    print(f"{c:34}{res_tr['dice_per_class'].get(c, float('nan')):>9.4f}"
          f"{res_va['dice_per_class'].get(c, float('nan')):>9.4f}"
          f"{res_te['dice_per_class'].get(c, float('nan')):>9.4f}"
          f"{res_te['n_scored_per_class'][c]:>9}")
print(f"{'mean over foreground classes':34}{res_tr['mean_foreground_dice']:>9.4f}"
      f"{res_va['mean_foreground_dice']:>9.4f}{res_te['mean_foreground_dice']:>9.4f}")

print("\nBraTS composite regions (what the literature reports)\n")
print(f"{'region':20}{'train':>9}{'val':>9}{'test':>9}{'test sd':>10}")
for r in ("whole_tumour", "tumour_core", "enhancing"):
    print(f"{r:20}{res_tr['dice_composite'].get(r, float('nan')):>9.4f}"
          f"{res_va['dice_composite'].get(r, float('nan')):>9.4f}"
          f"{res_te['dice_composite'].get(r, float('nan')):>9.4f}"
          f"{res_te['composite_std'].get(r, float('nan')):>10.4f}")

wt = [r["whole_tumour"] for r in res_te["per_patient"] if r["whole_tumour"] is not None]
print(f"\nper-patient whole-tumour Dice on test, sorted: "
      f"{', '.join(f'{v:.3f}' for v in sorted(wt))}")
print(f"worst patient {min(wt):.3f}, median {np.median(wt):.3f}, best {max(wt):.3f}")
print("-> the mean alone would hide this range, which is why it is printed")

results = {"task": "BraTS 2020 brain tumour segmentation",
           "model": f"3D U-Net, {len(USE_MODS)} channels {USE_MODS}, base 16, depth 4, "
                    f"trained from scratch",
           "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 3),
           "grid": GRID, "patch": PATCH, "epochs": EPOCHS, "batch_size": BS,
           "split": "by patient, 80/15/5, seed 0",
           "n_patients": {"train": len(tr_c), "val": len(va_c), "test": len(te_c)},
           "classes": CLASSES,
           "selection": "epoch chosen by validation Dice; test patients untouched",
           "caveat_resolution": f"Dice measured on a {GRID}^3 resample of the "
                                f"brain-cropped volume, not the native 240x240x155 grid",
           "train": {k: v for k, v in res_tr.items() if k != "per_patient"},
           "val": {k: v for k, v in res_va.items() if k != "per_patient"},
           "test": res_te, "history": hist}
with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=1)
print("\nwrote /kaggle/working/results.json")

## Curves and the per-patient spread

In [ ]:
import matplotlib.pyplot as plt

ep = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), dpi=130)
ax[0].plot(ep, [h["train_loss"] for h in hist], lw=1.6)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("train loss")
ax[0].set_title("training loss (Dice + weighted CE)", fontsize=10)
v = [(h["epoch"], h["val_composite"]) for h in hist if "val_composite" in h]
for r, col in (("whole_tumour", "#3b6ea5"), ("tumour_core", "#c87137"),
               ("enhancing", "#5a9367")):
    ax[1].plot([a for a, _ in v], [b.get(r, np.nan) for _, b in v], "o-", lw=1.5,
               ms=3.5, color=col, label=r.replace("_", " "))
    ax[1].scatter([ep[-1]], [res_te["dice_composite"].get(r, np.nan)], marker="*",
                  s=120, color=col, zorder=5)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("Dice")
ax[1].set_title("validation Dice by region (star = final test)", fontsize=10)
ax[1].legend(fontsize=8, loc="lower right")
for a in ax:
    a.grid(alpha=0.25, lw=0.5); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# Per-patient distribution on test. A box plot of 3 regions says more than 3 means.
fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=130)
data = [[r[k] for r in res_te["per_patient"] if r[k] is not None]
        for k in ("whole_tumour", "tumour_core", "enhancing")]
ax.boxplot(data, widths=0.5)          # 'labels=' was renamed in matplotlib 3.9
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["whole\ntumour", "tumour\ncore", "enhancing"])
for i, d in enumerate(data):
    ax.scatter(np.full(len(d), i + 1) + np.random.uniform(-0.1, 0.1, len(d)), d,
               s=14, alpha=0.65, color="#3b6ea5", zorder=3)
ax.set_ylabel("Dice"); ax.set_ylim(0, 1)
ax.set_title(f"per-patient Dice, {len(te_c)} held-out test patients", fontsize=10)
ax.grid(alpha=0.25, lw=0.5, axis="y"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## The worst, median and best test patient

In [ ]:
# Show the best and the WORST test patient, at the slice with the most tumour.
order = sorted([r for r in res_te["per_patient"] if r["whole_tumour"] is not None],
               key=lambda r: r["whole_tumour"])
picks = [(order[0], "worst"), (order[len(order) // 2], "median"), (order[-1], "best")]
model.eval()
fig, axes = plt.subplots(len(picks), 3, figsize=(9.5, 3.2 * len(picks)), dpi=130)
for r, (row, tag) in enumerate(picks):
    d = np.load(os.path.join(CACHE, f"{row['case']}.npz"))
    y = d["y"]
    with torch.no_grad(), torch.cuda.amp.autocast():
        pr = model(torch.from_numpy(d["x"].astype("float32"))[None].to(DEV)
                   ).float().argmax(1)[0].cpu().numpy()
    z = int(np.argmax([(y[:, :, k] > 0).sum() for k in range(y.shape[2])]))
    img = d["x"][0, :, :, z].astype("float32")
    for c, (m, t) in enumerate(((None, f"{USE_MODS[0]} MRI"), (y[:, :, z], "annotation"),
                                (pr[:, :, z], "prediction"))):
        a = axes[r, c]
        a.imshow(np.rot90(img), cmap="gray")
        if m is not None:
            mm = np.rot90(m).astype("float32"); mm[mm == 0] = np.nan
            a.imshow(mm, cmap="tab10", vmin=0, vmax=9, alpha=0.65, interpolation="nearest")
        a.set_xticks([]); a.set_yticks([])
        if r == 0:
            a.set_title(t, fontsize=10)
        if c == 0:
            a.set_ylabel(f"{tag}\nWT Dice {row['whole_tumour']:.3f}", fontsize=8)
plt.tight_layout(); plt.show()

## What this shows, and what it does not

A 3D U-Net trained from scratch on BraTS 2020 reaches the per-class and per-region Dice
printed above on patients it never saw, with the epoch chosen on a separate validation
split.

The limits of this result, stated plainly:

- **The resolution is not the native one.** Every volume was cropped to the brain and
  resampled to a 128³ grid before training, and Dice is computed on that grid. Published
  BraTS numbers are computed at 240×240×155. The two are not directly comparable, and the
  resampling is likely to flatter small structures slightly, because a thin boundary that
  one grid resolves and the other does not contributes less surface error.
- **The test split is small.** Five percent of 369 patients is around 19 cases. A
  per-class mean from 19 cases has a wide interval, which is why the per-patient values
  are printed and plotted rather than summarised into a single number.
- **Enhancing tumour is the honest weak point.** It is the smallest class, it is absent
  entirely in some patients, and cases where it is absent are excluded from its Dice
  rather than scored as a free 1.0. That makes its number lower than a convention that
  counts correct absences as perfect, and more informative.
- **No comparison is run.** This is one architecture at one budget. It establishes nothing
  about nnU-Net, Swin UNETR, or a 2D baseline, because none of those were trained here.
  A single training run also carries seed variance that one run cannot measure.
- **Dice is not clinical utility.** It is overlap. It is insensitive to whether the error
  sits somewhere that would change a treatment decision.

Part of a portfolio: [github.com/hammasbuilds](https://github.com/hammasbuilds) ·
[vision-portfolio-woad.vercel.app](https://vision-portfolio-woad.vercel.app)
